In [12]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad, quad

In [13]:
# experimental data
save_folder = 'run7'
n_points = 5000

lower_factor = 0.99
upper_factor = 2 - lower_factor

# Load experimental data
atlas_data = pd.read_csv('../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_12500/1048597570.py:9: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead

/tmp/ipykernel_12500/1048597570.py:10: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



In [14]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [15]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q2, phi, mg, a1, a2, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q2, phi, mg, a1, a2, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  


def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [16]:
n_points = 1750

def full_int(mg, a1, a2, m2_func, q2_val, sqrt_s):
    # Garante que q_val seja array 1D
    q2_val = np.atleast_1d(q2_val)
    results = []

    for q2 in q2_val:
        def integrand(y, x, mg, a1, a2, m2_func, q2_val):
            k = sqrt_s * x
            phi = 2 * np.pi * y
            jacobian = 2 * np.pi * sqrt_s
            result = k * (
                T_1(k, q2_val, phi, mg, a1, a2, m2_func)
                - T_2(k, q2_val, phi, mg, a1, a2, m2_func)
            ) * jacobian
            return result

        def inner_integral(x):
            integral_real = fixed_quad(
                lambda y: np.real(integrand(y, x, mg, a1, a2, m2_func, q2)),
                0, 1, n=n_points
            )[0]
            integral_imag = fixed_quad(
                lambda y: np.imag(integrand(y, x, mg, a1, a2, m2_func, q2)),
                0, 1, n=n_points
            )[0]
            return integral_real + 1j * integral_imag

        integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
        results.append(integral_value)

    # Retorna escalar se apenas um q_val foi passado
    return np.array(results) if len(results) > 1 else results[0]

In [17]:
# eps_min = 0.0616
# mg_min = 0.389
# a1_min = 1.50	
# a2_min = 2.13

eps_min = 0.1
mg_min = 0.814
a1_min = 1.295	
a2_min = 1.0

In [18]:
# Calculates and plot dif sigma 
lst_amp_born_diff = []
def get_dif_sigma(epsilon, mg, a1, a2, mg_model):

    sqrt_s = 7000
    scale = 1  # caso único
    start_q2 = 1e-2
    max_q2   = 0.2001
    q2_step  = 0.001


    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2
        integral_value = full_int(
                mg, a1, a2, mg_model, q2, sqrt_s
        )
        # print(integral_value)
        
        diff_T = integral_value


        s          = sqrt_s**2
        amp_value  = amp_calculation(diff_T, s, epsilon, t)
        lst_amp_born_diff.append(amp_value)
        dif_sigma  = differential_sigma(amp_value, s) * scale
        # print(f"q2 = {q2}, diff_t = {integral_value}, amp born = {amp_value:6e} \n")

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step

    return {sqrt_s: (lst_q2, lst_dif_sigma)}

#for pl atlas
dif_sigma_pl_atlas = get_dif_sigma(
    eps_min,
    mg_min,
    a1_min,
    a2_min,
    m2_pl
)

dif_sigma_pl_atlas_7_q2 = dif_sigma_pl_atlas[7000][0]
dif_sigma_pl_atlas_7_values = dif_sigma_pl_atlas[7000][1]


In [19]:


def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))

fig_atlas = go.Figure()


# for pl atlas
add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, dif_sigma_pl_atlas_7_values,label='7 TeV', color='blue', mg_model='pl')

#data points
add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# Atualiza layout
fig_atlas.update_layout(
    title='dσ/dt vs. |t| - Log and PL models in ATLAS',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Mass Model',
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')

# fig_atlas.show(renderer='browser')


In [20]:
import numpy as np
from scipy.special import j0
from scipy.integrate import quad

lst_chi = []
lst_amp_eik = []
lst_diff_sigma = []


sqrt_s = 7000
s = sqrt_s ** 2

eps_rel = 1e-6
eps_abs = 1e-16

q_max_chi = 1.0          # limite de q na Eq. 23
q_max_eik = np.sqrt(0.1) # limite de q (momentum transfer) na Eq. 24
b_max     = 10.0

limit = 100

# ── Eq. 23: χ(s,b) como integral direta em q ─────────────────────────────────
def chi(b_val):
    def integrand_real(q_val):
        q2_val = q_val ** 2
        t = -q2_val
        diff_t = full_int(mg_min, a1_min, a2_min, m2_pl, q2_val, sqrt_s)
        born_amp = amp_calculation(diff_t, s, eps_min, t)
        return np.real((1/s) * q_val * j0(b_val * q_val) * born_amp)

    def integrand_imag(q_val):
        q2_val = q_val ** 2
        t = -q2_val
        diff_t = full_int(mg_min, a1_min, a2_min, m2_pl, q2_val, sqrt_s)
        born_amp = amp_calculation(diff_t, s, eps_min, t)
        return np.imag((1/s) * q_val * j0(b_val * q_val) * born_amp)

    real_part, _ = quad(integrand_real, 0, q_max_chi, epsrel=eps_rel, epsabs=eps_abs, limit=limit)
    imag_part, _ = quad(integrand_imag, 0, q_max_chi, epsrel=eps_rel, epsabs=eps_abs, limit=limit)
    return real_part + 1j * imag_part

# ── Eq. 24: A_eik(s,t) — integral em b com χ(b) calculado internamente ───────
lst_q_ext = np.linspace(0, q_max_eik, 30)

for q_ext in lst_q_ext:

    def integrand_real(b_val):
        chi_val = chi(b_val)
        return np.real(b_val * j0(q_ext * b_val) * (1 - np.exp(1j * chi_val)))

    def integrand_imag(b_val):
        chi_val = chi(b_val)
        return np.imag(b_val * j0(q_ext * b_val) * (1 - np.exp(1j * chi_val)))

    real_part, _ = quad(integrand_real, 0, b_max, epsrel=eps_rel, epsabs=eps_abs, limit=limit)
    imag_part, _ = quad(integrand_imag, 0, b_max, epsrel=eps_rel, epsabs=eps_abs, limit=limit)

    amp_eik = 1j * s * (real_part + 1j * imag_part)
    lst_amp_eik.append(amp_eik)

    diff_sigma_eik = (amp_eik.imag * amp_eik.imag) * (np.pi/s**2) * 0.389379323
    lst_diff_sigma.append(diff_sigma_eik)

    t_val = -q_ext**2
    print(f"  q2 = {q_ext**2:.4f}, A_eik = {amp_eik:.6e}")
    


  q2 = 0.0000, A_eik = 0.000000e+00+1.209574e+09j
  q2 = 0.0001, A_eik = 0.000000e+00+1.208341e+09j
  q2 = 0.0005, A_eik = 0.000000e+00+1.204647e+09j
  q2 = 0.0011, A_eik = 0.000000e+00+1.198510e+09j
  q2 = 0.0019, A_eik = 0.000000e+00+1.189958e+09j
  q2 = 0.0030, A_eik = 0.000000e+00+1.179031e+09j
  q2 = 0.0043, A_eik = 0.000000e+00+1.165780e+09j
  q2 = 0.0058, A_eik = 0.000000e+00+1.150266e+09j
  q2 = 0.0076, A_eik = 0.000000e+00+1.132561e+09j
  q2 = 0.0096, A_eik = 0.000000e+00+1.112746e+09j
  q2 = 0.0119, A_eik = 0.000000e+00+1.090912e+09j
  q2 = 0.0144, A_eik = 0.000000e+00+1.067157e+09j
  q2 = 0.0171, A_eik = 0.000000e+00+1.041591e+09j
  q2 = 0.0201, A_eik = 0.000000e+00+1.014327e+09j
  q2 = 0.0233, A_eik = 0.000000e+00+9.854866e+08j
  q2 = 0.0268, A_eik = 0.000000e+00+9.551986e+08j
  q2 = 0.0304, A_eik = 0.000000e+00+9.235952e+08j
  q2 = 0.0344, A_eik = 0.000000e+00+8.908138e+08j
  q2 = 0.0385, A_eik = 0.000000e+00+8.569952e+08j
  q2 = 0.0429, A_eik = 0.000000e+00+8.222830e+08j


In [21]:

# ── Plots ─────────────────────────────────────────────────────────────────────
import plotly.graph_objects as go

# Plot χ(b) avaliado nos pontos de b usados internamente (opcional, para diagnóstico)
b_plot = np.linspace(0, b_max, 50)
chi_plot = np.array([chi(b) for b in b_plot])

# fig1 = go.Figure()
# fig1.add_trace(go.Scatter(
#     x=b_plot, y=np.imag(chi_plot),
#     mode='lines+markers', name='Im χ(b)',
#     marker=dict(symbol='square', size=5, color='tomato'),
#     line=dict(color='tomato', width=2)
# ))
# fig1.update_layout(
#     title=dict(text="Eikonal phase χ(b) vs impact parameter b", font=dict(size=16)),
#     xaxis=dict(title_text="b", gridcolor='#aaaaaa'),
#     yaxis=dict(title_text="Im χ(b)", gridcolor='#aaaaaa'),
#     template="plotly_white", height=500, width=800
# )
# fig1.show()

q2_vals = lst_q_ext**2

# fig2 = go.Figure()
# fig2.add_trace(go.Scatter(
#     x=q2_vals, y=np.abs(np.imag(lst_amp_eik)),
#     mode='lines+markers', name='Im A_eik',
#     line=dict(color='tomato', width=2), marker=dict(size=5)
# ))
# fig2.update_xaxes(title_text="q² [GeV²]", gridcolor='#aaaaaa')
# fig2.update_yaxes(title_text="|Im A<sub>eik</sub>|", gridcolor='#aaaaaa', type='log')
# fig2.update_layout(
#     title=dict(text="Eikonalized amplitude A<sub>eik</sub>(s,q²)", font=dict(size=16)),
#     template="plotly_white", height=500, width=800
# )
# fig2.show()
# fig2.write_image('amp_eik_plot.pdf', width=1200, height=600)
# fig2.write_html('amp_eik_plot.html')

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=q2_vals, y=lst_diff_sigma,
    mode='lines+markers', name='diff_sigma',
    line=dict(color='blue', width=2), marker=dict(size=5)
))
add_data_trace(fig3, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

fig3.update_xaxes(title_text="q² [GeV²]", gridcolor='#aaaaaa')
fig3.update_yaxes(title_text="diff sigma", gridcolor='#aaaaaa', type='log')
fig3.update_layout(
    title=dict(text="Eikonalized diff sigma", font=dict(size=16)),
    template="plotly_white", height=500, width=800
)
fig3.show()
# fig3.write_image('diff_sigma_eik_plot.pdf', width=1200, height=600)
# fig3.write_html('diff_sigma_eik_plot.html')